## 1. Configuración Inicial y Preparación de Datos

In [ ]:
!pip install roboflow -q

import os
import glob
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from roboflow import Roboflow
from torch.utils.data import DataLoader, Dataset
from torchvision.ops import generalized_box_iou_loss
from torchvision import transforms
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

### 1.1. Estructura del Dataset
Definición de las rutas para entrenamiento, validación y prueba.
Se espera que cada partición contenga las imágenes (.jpg) y sus respectivas etiquetas (.txt).

In [ ]:
rf = Roboflow(api_key="9smAmQgaD8pNOTsDMYKR")
project = rf.workspace("roboflow-100").project("furniture-ngpea")
version = project.version(2)
dataset = version.download("yolov11")

# Placeholder de Rutas
data_dir = "./dataset_det"
train_img_dir = os.path.join(data_dir, "images", "train")
train_label_dir = os.path.join(data_dir, "labels", "train")

NUM_CLASSES = 3

### 1.2. Dataset y Collate Function
Definimos un Dataset optimizado con caching de etiquetas YOLO en memoria (evitando latencia de disco)
y una función de empaquetado (collate_fn) que indexa los batch para la función de pérdida.

In [ ]:
class YOLODataset(Dataset):
    """
    Dataset para Detección de Objetos en formato YOLO (.txt).
    Carga todas las etiquetas en memoria durante la inicialización para máxima eficiencia I/O.
    """
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform
        
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        self.labels_cache = []
        
        # Optimización: Caching en memoria de las etiquetas
        for img_path in self.img_paths:
            base_name = os.path.basename(img_path).replace('.jpg', '.txt')
            label_path = os.path.join(label_dir, base_name)
            
            if os.path.exists(label_path):
                # Se espera formato YOLO: [clase, x_center, y_center, w, h]
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    if len(lines) > 0:
                        boxes = np.array([list(map(float, line.strip().split())) for line in lines])
                    else:
                        boxes = np.empty((0, 5))
            else:
                boxes = np.empty((0, 5))
            
            self.labels_cache.append(torch.tensor(boxes, dtype=torch.float32))

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path).convert("RGB")
        boxes = self.labels_cache[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, boxes

def yolo_collate_fn(batch):
    """
    Collate function personalizada.
    Añade una columna inicial a cada tensor de etiquetas que indica su índice en el batch.
    Esto permite que la DetectionLoss filtre las etiquetas usando máscaras booleanas.
    """
    images, labels = zip(*batch)
    images = torch.stack(images, 0)
    
    batched_labels = []
    for batch_idx, boxes in enumerate(labels):
        if boxes.size(0) > 0:
            # Creamos un tensor del tamaño [num_boxes, 1] lleno del índice actual
            idx_tensor = torch.full((boxes.size(0), 1), batch_idx, dtype=boxes.dtype)
            # Concatenamos: resultado final -> [batch_idx, class_id, x, y, w, h]
            boxes_with_idx = torch.cat((idx_tensor, boxes), dim=1)
            batched_labels.append(boxes_with_idx)
            
    if len(batched_labels) > 0:
        batched_labels = torch.cat(batched_labels, 0)
    else:
        batched_labels = torch.empty((0, 6))
        
    return images, batched_labels

# Transformación base 
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
])

try:
    train_dataset = YOLODataset(train_img_dir, train_label_dir, transform)
    
    # Configuración Global de Dataloader para máxima eficiencia de I/O
    train_loader = DataLoader(
        train_dataset, 
        batch_size=8, 
        shuffle=True, 
        collate_fn=yolo_collate_fn,
        num_workers=4,         # Multi-threading para carga
        pin_memory=True,       # Transferencia rápida a GPU
        prefetch_factor=2      # Precarga proactiva de batches
    )
    print("DataLoader de detección listo y optimizado.")
except FileNotFoundError:
    print("Advertencia: Carpetas no encontradas. Entorno configurado para ejecución secuencial sin datos.")
    train_loader = None

## 2. Arquitectura y Lógica de Entrenamiento

### 2.1. Modelo inspirado en YOLOv11
Definimos una arquitectura con bloques tipo C3k2, un Neck PANet simulado y un Head Desacoplado.

In [ ]:
class C3k2Block(nn.Module):
    """
    Bloque CSP Bottleneck moderno.
    Mejora el flujo de gradientes limitando los parámetros respecto a convoluciones estándar.
    """
    def __init__(self, c1, c2):
        super().__init__()
        c_ = c2 // 2
        self.cv1 = nn.Conv2d(c1, c_, 1, 1)
        self.cv2 = nn.Conv2d(c1, c_, 1, 1)
        self.cv3 = nn.Conv2d(c_, c_, 3, 1, padding=1)
        self.cv4 = nn.Conv2d(2 * c_, c2, 1, 1)
        self.act = nn.SiLU()

    def forward(self, x):
        y1 = self.cv3(self.cv1(x))
        y2 = self.cv2(x)
        return self.cv4(self.act(torch.cat((y1, y2), 1)))

class DecoupledHead(nn.Module):
    """
    Cabezal desacoplado: Separa las ramas de regresión espacial y clasificación.
    Fundamental para mejorar el Task-Aligned Assignment y acelerar convergencia.
    """
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.stem = nn.Conv2d(in_channels, in_channels, 1, 1)
        
        # Rama Regresión (coordenadas x, y, w, h y objectness)
        self.reg_conv = nn.Conv2d(in_channels, in_channels, 3, 1, padding=1)
        self.reg_pred = nn.Conv2d(in_channels, 5, 1, 1)
        
        # Rama Clasificación
        self.cls_conv = nn.Conv2d(in_channels, in_channels, 3, 1, padding=1)
        self.cls_pred = nn.Conv2d(in_channels, num_classes, 1, 1)

    def forward(self, x):
        x = self.stem(x)
        reg = self.reg_pred(self.reg_conv(x))
        cls = self.cls_pred(self.cls_conv(x))
        return torch.cat([reg, cls], dim=1) # Shape: [B, 5+C, H, W]

class YOLOv11Base(nn.Module):
    """
    Arquitectura de detección con Backbone modular, Neck PANet y Decoupled Head.
    """
    def __init__(self, num_classes=80):
        super().__init__()
        # Backbone 
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.SiLU(),
            C3k2Block(32, 64),
            nn.MaxPool2d(2)
        )
        # Neck (PANet Base)
        self.neck = C3k2Block(64, 128)
        # Head
        self.head = DecoupledHead(128, num_classes)

    def forward(self, x):
        features = self.neck(self.backbone(x))
        return self.head(features)


model = YOLOv11Base(num_classes=NUM_CLASSES).to(device)

### 2.2. Implementación de Detection Loss
Adaptación del loss de YOLO para tensores de dimensiones espaciales (N, 6).

In [ ]:
def cxcywh_to_xyxy(boxes):
    """
    Función auxiliar: Convierte formato [cx, cy, w, h] a [x1, y1, x2, y2].
    Requerido para el cálculo de la métrica IoU.
    """
    x_c, y_c, w, h = boxes.unbind(-1)
    x1 = x_c - 0.5 * w
    y1 = y_c - 0.5 * h
    x2 = x_c + 0.5 * w
    y2 = y_c + 0.5 * h
    return torch.stack([x1, y1, x2, y2], dim=-1)

class DetectionLoss(nn.Module):
    """
    Función de pérdida funcional. 
    Implementa Label Assignment buscando la celda responsable,
    GIoU Loss para cajas y BCEWithLogits para objectness y clases.
    """
    def __init__(self, num_classes=80):
        super().__init__()
        self.num_classes = num_classes
        self.bce = nn.BCEWithLogitsLoss()
        
    def forward(self, preds, targets):
        """
        preds: Tensor [B, 5+C, H, W] -> (tx, ty, tw, th, obj, cls...)
        targets: Tensor [N, 6] -> [batch_idx, cls, cx, cy, w, h] normalizados entre 0 y 1.
        """
        B, channels, H, W = preds.shape
        
        # Reorganizar preds para acceder por coordenadas espaciales: [B, H, W, 5+C]
        preds = preds.permute(0, 2, 3, 1)
        
        # 1. Inicializar tensores objetivo (Todo asume ser "fondo" inicialmente)
        t_obj = torch.zeros((B, H, W, 1), device=preds.device)
        t_cls = torch.zeros((B, H, W, self.num_classes), device=preds.device)
        t_box = torch.zeros((B, H, W, 4), device=preds.device)
        
        # Máscara booleana espacial para identificar celdas con objetos
        obj_mask = torch.zeros((B, H, W), dtype=torch.bool, device=preds.device)
        
        # 2. Iteración sobre cada imagen (Filtrado de etiquetas por índice)
        for b_idx in range(B):
            mask = targets[:, 0] == b_idx
            img_targets = targets[mask]
            
            for t in img_targets:
                cls_id = int(t[1].item())
                cx, cy, w, h = t[2:6] 
                
                # Proyectar el centro (cx, cy) normalizado a los índices de la grilla HxW
                gi = int(cx * W)
                gj = int(cy * H)
                
                # Prevenir desbordamientos en los bordes
                gi = min(max(gi, 0), W - 1)
                gj = min(max(gj, 0), H - 1)
                
                # Asignar la "responsabilidad" de predecir este target a la celda (gj, gi)
                obj_mask[b_idx, gj, gi] = True
                t_obj[b_idx, gj, gi, 0] = 1.0
                t_cls[b_idx, gj, gi, cls_id] = 1.0
                t_box[b_idx, gj, gi] = torch.tensor([cx, cy, w, h], device=preds.device)

        # --- CÁLCULO DE LOSS REAL ---
        
        # A) Objectness Loss (Se evalúa en TODA la grilla para penalizar falsos positivos)
        pred_obj = preds[..., 4:5]
        obj_loss = self.bce(pred_obj, t_obj)
        
        box_loss = torch.tensor(0.0, device=preds.device)
        cls_loss = torch.tensor(0.0, device=preds.device)
        
        # B) Box & Class Loss (Se evalúa SOLO en las celdas responsables del objeto)
        if obj_mask.sum() > 0:
            pred_box = preds[..., 0:4][obj_mask]  
            pred_cls = preds[..., 5:][obj_mask]
            
            t_box_obj = t_box[obj_mask]
            t_cls_obj = t_cls[obj_mask]
            
            # Simulación de las activaciones de la head para las coordenadas espaciales
            # (Asumiendo que el modelo arroja logits crudos)
            pred_cxcy = torch.sigmoid(pred_box[..., 0:2]) 
            pred_wh = torch.exp(pred_box[..., 2:4]) 
            pred_box_norm = torch.cat([pred_cxcy, pred_wh], dim=-1)
            
            # Convertir formato para GIoU Loss de torchvision
            pred_xyxy = cxcywh_to_xyxy(pred_box_norm)
            t_xyxy = cxcywh_to_xyxy(t_box_obj)
            
            # Cálculo exacto de pérdidas de caja (GIoU) y clase (BCE)
            box_loss = generalized_box_iou_loss(pred_xyxy, t_xyxy, reduction="mean")
            cls_loss = self.bce(pred_cls, t_cls_obj)

        # C) Suma Ponderada Global
        total_loss = (1.5 * box_loss) + (0.5 * cls_loss) + (1.0 * obj_loss)
        
        return total_loss

criterion = DetectionLoss(num_classes=NUM_CLASSES)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

## 3. Monitoreo y Evaluación

### 3.1. Métricas con NMS y Reportes por Clase
Tras aplicar la Supresión de No-Máximos (NMS), se consolidan mAP50 y Accuracy por clase.

In [ ]:
def cxcywh_to_xyxy_flat(boxes):
    """Convierte [cx, cy, w, h] a [x1, y1, x2, y2] para tensores 2D."""
    x_c, y_c, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    return torch.stack([x_c - w/2, y_c - h/2, x_c + w/2, y_c + h/2], dim=1)

def real_nms(predictions, conf_thres=0.25, iou_thres=0.45):
    """
    Non-Maximum Suppression (NMS) real.
    Convierte los mapas de características [B, 5+C, H, W] en cajas finales filtradas.
    Retorna una lista de tensores [N_cajas, 6] con formato (x1, y1, x2, y2, conf, cls).
    """
    B, C, H, W = predictions.shape
    # Aplanar dimensiones espaciales: [B, H*W, 5+C]
    preds = predictions.view(B, C, -1).permute(0, 2, 1)
    
    output = [torch.zeros((0, 6), device=predictions.device)] * B
    
    for i, image_pred in enumerate(preds):
        # 1. Aplicar activaciones asumiendo logits crudos de la Head
        # Cajas (cx, cy normalizadas con sigmoid, w, h con exp) - Aproximación general
        boxes = image_pred[:, :4]
        boxes[:, :2] = torch.sigmoid(boxes[:, :2])
        boxes[:, 2:4] = torch.exp(boxes[:, 2:4])
        boxes_xyxy = cxcywh_to_xyxy_flat(boxes)
        
        # Objectness y clases
        obj_conf = torch.sigmoid(image_pred[:, 4])
        cls_conf = torch.sigmoid(image_pred[:, 5:])
        
        # 2. Filtrado inicial por confianza (Objectness * Max Class Conf)
        max_cls_conf, max_cls_id = torch.max(cls_conf, dim=1)
        conf = obj_conf * max_cls_conf
        
        mask = conf > conf_thres
        if not mask.any():
            continue
            
        filtered_boxes = boxes_xyxy[mask]
        filtered_conf = conf[mask]
        filtered_cls = max_cls_id[mask].float()
        
        # 3. Aplicar NMS real de torchvision
        keep_indices = torchvision.ops.nms(filtered_boxes, filtered_conf, iou_thres)
        
        # 4. Ensamblar tensor final para la imagen: [x1, y1, x2, y2, conf, cls]
        final_dets = torch.cat((
            filtered_boxes[keep_indices], 
            filtered_conf[keep_indices].unsqueeze(1), 
            filtered_cls[keep_indices].unsqueeze(1)
        ), dim=1)
        
        output[i] = final_dets
        
    return output

def calculate_metrics_per_batch(nms_outputs, targets, num_classes, iou_threshold=0.5):
    """
    Calcula TP, FP y FN por clase comparando las detecciones post-NMS 
    contra los targets usando torchvision.ops.box_iou.
    """
    tp = {c: 0 for c in range(num_classes)}
    fp = {c: 0 for c in range(num_classes)}
    fn = {c: 0 for c in range(num_classes)}
    
    B = len(nms_outputs)
    
    for b_idx in range(B):
        preds = nms_outputs[b_idx]  # [N_preds, 6] -> xyxy, conf, cls
        mask = targets[:, 0] == b_idx
        img_targets = targets[mask] # [N_targets, 6] -> idx, cls, cx, cy, w, h
        
        # Extraer y convertir cajas target a xyxy
        if len(img_targets) > 0:
            target_cls = img_targets[:, 1]
            target_boxes = cxcywh_to_xyxy_flat(img_targets[:, 2:6])
        else:
            target_cls = torch.tensor([], device=preds.device)
            target_boxes = torch.tensor([], device=preds.device)
        
        # Si no hay predicciones pero hay targets, todos son Falsos Negativos
        if len(preds) == 0:
            for c in target_cls:
                fn[int(c.item())] += 1
            continue
            
        pred_boxes = preds[:, :4]
        pred_cls = preds[:, 5]
        
        # Si hay predicciones pero no targets, todos son Falsos Positivos
        if len(target_boxes) == 0:
            for c in pred_cls:
                fp[int(c.item())] += 1
            continue
            
        # Calcular matriz de IoU entre predicciones y targets [N_preds, N_targets]
        iou_matrix = torchvision.ops.box_iou(pred_boxes, target_boxes)
        
        matched_targets = set()
        
        # Evaluación Greedy (Aproximación para mAP@0.5 y Accuracy)
        for p_idx, p_box in enumerate(pred_boxes):
            p_c = int(pred_cls[p_idx].item())
            
            # Buscar el mejor target para esta predicción
            best_iou = 0
            best_t_idx = -1
            
            for t_idx, t_box in enumerate(target_boxes):
                if int(target_cls[t_idx].item()) == p_c and t_idx not in matched_targets:
                    iou = iou_matrix[p_idx, t_idx].item()
                    if iou > best_iou:
                        best_iou = iou
                        best_t_idx = t_idx
            
            if best_iou >= iou_threshold:
                tp[p_c] += 1
                matched_targets.add(best_t_idx)
            else:
                fp[p_c] += 1
                
        # Los targets no emparejados son Falsos Negativos
        for t_idx, t_c in enumerate(target_cls):
            if t_idx not in matched_targets:
                fn[int(t_c.item())] += 1

    return tp, fp, fn

def report_final_metrics(tp, fp, fn, num_classes):
    """
    Agrega los contadores de TP, FP y FN de toda la época para calcular 
    Accuracy y una aproximación del mAP@0.5.
    """
    class_accs = {}
    class_aps = {}
    
    for c in range(num_classes):
        t = tp[c]
        f_p = fp[c]
        f_n = fn[c]
        
        # Accuracy: TP / (TP + FP + FN)
        denom_acc = t + f_p + f_n
        class_accs[c] = t / denom_acc if denom_acc > 0 else 0.0
        
        # Precision: TP / (TP + FP) (Aproximación de AP al 50%)
        denom_prec = t + f_p
        class_aps[c] = t / denom_prec if denom_prec > 0 else 0.0

    macro_acc = np.mean(list(class_accs.values()))
    map50 = np.mean(list(class_aps.values()))
    
    print("\n" + "="*45)
    print(f"REPORTE DETALLADO - EVALUACIÓN REAL")
    print("="*45)
    print(f"Métricas Generales (IoU > 0.5):")
    print(f"  > mAP@0.50 (Aprox) : {map50:.4f}")
    print(f"  > Macro Avg ACC    : {macro_acc:.4f}\n")
    print("Desglose de Accuracy por Clase:")
    for c in range(num_classes):
        print(f"  - Clase {c}        : {class_accs[c]:.4f}")
    print("="*45)
    
    return map50, macro_acc, class_accs

### 3.2. Ciclo de Entrenamiento
Bucle principal para entrenar y validar el modelo durante múltiples épocas.

In [ ]:
def train_detection(model, train_loader, val_loader, optimizer, criterion, epochs=2, num_classes=80):
    device = next(model.parameters()).device
    
    for epoch in range(epochs):
        # --- FASE DE ENTRENAMIENTO ---
        model.train()
        train_loss = 0.0
        
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            
            preds = model(images)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        
        # --- FASE DE VALIDACIÓN ---
        model.eval()
        epoch_tp = {c: 0 for c in range(num_classes)}
        epoch_fp = {c: 0 for c in range(num_classes)}
        epoch_fn = {c: 0 for c in range(num_classes)}
        
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(device), targets.to(device)
                
                preds = model(images)
                # 1. Aplicar NMS Real
                nms_outputs = real_nms(preds, conf_thres=0.25, iou_thres=0.45)
                
                # 2. Calcular TP/FP/FN del batch actual
                batch_tp, batch_fp, batch_fn = calculate_metrics_per_batch(
                    nms_outputs, targets, num_classes
                )
                
                # Acumular contadores para la época
                for c in range(num_classes):
                    epoch_tp[c] += batch_tp[c]
                    epoch_fp[c] += batch_fp[c]
                    epoch_fn[c] += batch_fn[c]
        
        print(f"\nÉpoca [{epoch+1}/{epochs}] completada. Train Loss: {avg_train_loss:.4f}")
        
        # 3. Reporte Real de la época
        report_final_metrics(epoch_tp, epoch_fp, epoch_fn, num_classes)

# Para ejecutar el entrenamiento (requiere dataset físico), descomentar:
# train_detection(model, train_loader, valid_loader, criterion, optimizer, epochs=5, num_classes=NUM_CLASSES) 

## 4. Evaluación Final

### 4.1. Evaluación en el Conjunto de Prueba
Se extraen las métricas finales (mAP50, Macro Avg Acc y detalle por clase) utilizando datos nunca antes vistos.

In [ ]:
def evaluate_test_set(model, test_loader, num_classes, conf_thres=0.25, iou_thres=0.45):
    """
    Aplica el reporte detallado estandarizado al set de prueba.
    Garantiza integridad total realizando inferencia pura (no_grad)
    y calculando las métricas globales reales acumulando TP/FP/FN.
    """
    if test_loader is None:
        print("Test DataLoader no disponible para evaluación final.")
        return
        
    device = next(model.parameters()).device
    model.eval()
    
    # Inicializar contadores globales para todo el set de prueba
    test_tp = {c: 0 for c in range(num_classes)}
    test_fp = {c: 0 for c in range(num_classes)}
    test_fn = {c: 0 for c in range(num_classes)}
    
    print("\nIniciando inferencia en el Test Set...")
    
    with torch.no_grad():
        for images, targets in test_loader:
            images, targets = images.to(device), targets.to(device)
            
            # Inferencia pura
            preds = model(images)
            
            # 1. Post-procesamiento real (NMS)
            nms_outputs = real_nms(preds, conf_thres=conf_thres, iou_thres=iou_thres)
            
            # 2. Emparejamiento e intersección (IoU) del batch actual
            batch_tp, batch_fp, batch_fn = calculate_metrics_per_batch(
                nms_outputs, targets, num_classes
            )
            
            # 3. Acumulación global (Evita el sesgo por tamaño de batch)
            for c in range(num_classes):
                test_tp[c] += batch_tp[c]
                test_fp[c] += batch_fp[c]
                test_fn[c] += batch_fn[c]
                
    print("\n[RESULTADOS FINALES EN SET DE PRUEBA]")
    # Reutilizamos la función de reporte que ya hace las divisiones exactas
    report_final_metrics(test_tp, test_fp, test_fn, num_classes)

# Para evaluar en el conjunto de prueba, descomentar:
# evaluate_test_set(model, test_loader, num_classes=NUM_CLASSES)